# Attention Consistency λ2 Sweep — Phase 2 (Dinura / Person 3)

Tunes λ2 for the Attention Consistency Loss at full scale (3576/766/766, seed 42, 20 epochs). λ2=0.3 / MSE is **already done** by Kalana — seed it, then train the remaining cells.

Upload **only** `lambda_sweep.zip` (from `make_colab_zip.py`), not the whole repo.

**Before running:** Runtime → GPU (T4+). Upload the zip to `MyDrive/lambda_sweep.zip`.


## Step 0: Unzip + deps

Checkpoints/results write to `MyDrive/lambda_sweep_outputs` so a dropped runtime does not lose multi-hour runs.


In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/lambda_sweep.zip")
BUNDLE = Path("/content/lambda_sweep")
OUTPUTS = Path("/content/drive/MyDrive/lambda_sweep_outputs")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(f"Upload zip to {ZIP_ON_DRIVE}")
        print("Unzipping", ZIP_ON_DRIVE)
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("paths.py not found after unzip")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    if not (HERE / "paths.py").exists():
        cand = Path("Phase2/Dinura-Person3").resolve()
        if (cand / "paths.py").exists():
            HERE = cand
    OUTPUTS = HERE

sys.path.insert(0, str(HERE))
print("HERE =", HERE)


In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "thop", "tqdm", "opencv-python-headless"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")


In [ ]:
import paths
from paths import (
    LAYOUT_MODE, PERSON3_DIR, DEFAULT_LAMBDA2_SWEEP,
    add_teammate_paths, apply_data_dirs, set_output_roots,
)

set_output_roots(OUTPUTS / "checkpoints", OUTPUTS / "results")
add_teammate_paths()
apply_data_dirs()

print("layout:", LAYOUT_MODE)
print("attn pkg:", (PERSON3_DIR / "attention_consistency").is_dir())
print("images:", paths.DATA_IMG_DIR.is_dir(), paths.DATA_IMG_DIR)
print("masks:", paths.DATA_MASK_DIR.is_dir(), paths.DATA_MASK_DIR)
print("CKPT root:", paths.OUTPUT_ROOT_CKPT)
print("RESULTS root:", paths.OUTPUT_ROOT_RESULTS)
print("default lambda2 sweep:", DEFAULT_LAMBDA2_SWEEP)
assert (PERSON3_DIR / "attention_consistency").is_dir()
assert paths.DATA_IMG_DIR.is_dir() and paths.DATA_MASK_DIR.is_dir()


## Step 1: Device check


In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU — each att cell is ~2h on T4; CPU will not finish.")


## Step 2: Seed lambda2=0.3 / MSE from Kalana (metrics only)

If the zip includes Kalana results under `vendor/kalana_phase2/`, this copies them into `results/runs/l2_0.3_mse/`.


In [ ]:
import json, shutil
from paths import run_results_dir, run_tag, KALANA_PHASE2_DIR

src = KALANA_PHASE2_DIR / "results"
dst = run_results_dir(0.3, "mse")
needed = ["train_summary_att.json", "training_log_att.csv", "eval_att.json"]
if all((src / n).exists() for n in needed):
    dst.mkdir(parents=True, exist_ok=True)
    for n in needed:
        shutil.copy2(src / n, dst / n)
    summary = json.loads((dst / "train_summary_att.json").read_text())
    summary["source"] = "kalana_default_lambda2"
    summary["run_tag"] = run_tag(0.3, "mse")
    (dst / "train_summary_att.json").write_text(json.dumps(summary, indent=2))
    ev = json.loads((dst / "eval_att.json").read_text())
    ev.update(
        lambda2=0.3,
        att_mode="mse",
        run_tag=run_tag(0.3, "mse"),
        source="kalana_default_lambda2",
        model="SegFormer-B0 + Att (lambda2=0.3, mse)",
    )
    (dst / "eval_att.json").write_text(json.dumps(ev, indent=2))
    print("Seeded", dst)
else:
    print("Kalana metrics not in zip — skip seed.")
    print("looked for", src)


## Step 3: Train remaining MSE cells (0.1, 0.5, 1.0)

Skips any cell whose `segformer_b0_att_best.pt` already exists. ~2 hours per cell on T4.


In [ ]:
import run_lambda_sweep as S
import sys
sys.argv = [
    "run_lambda_sweep.py",
    "--lambda2", "0.1", "0.5", "1.0",
    "--att-mode", "mse",
]
S.main()


## Step 4: Eval MSE cells that have checkpoints

lambda2=0.3 seeded cell already has eval JSON — eval skips it without a local `.pt`. 0.1 / 0.5 / 1.0 get full test Dice/IoU/AAMO.


In [ ]:
import eval_lambda_sweep as E
import sys
sys.argv = [
    "eval_lambda_sweep.py",
    "--lambda2", "0.1", "0.3", "0.5", "1.0",
    "--att-mode", "mse",
]
E.main()


## Step 5 (optional): KL at the winning lambda2

After Step 4, read `winning_config.json`. Uncomment the block below to run KL.


In [ ]:
import json
win_path = paths.OUTPUT_ROOT_RESULTS / "winning_config.json"
WIN_L2 = None
if win_path.exists():
    w = json.loads(win_path.read_text()).get("winner") or {}
    if w.get("att_mode") == "mse":
        WIN_L2 = w.get("lambda2")
print("Suggested WIN_L2 for KL comparison:", WIN_L2)
# Uncomment to train+eval KL:
# if WIN_L2 is not None:
#     import run_lambda_sweep as S, eval_lambda_sweep as E, sys
#     sys.argv = ["run_lambda_sweep.py", "--lambda2", str(WIN_L2), "--att-mode", "kl"]
#     S.main()
#     sys.argv = ["eval_lambda_sweep.py", "--lambda2", str(WIN_L2), "--att-mode", "kl"]
#     E.main()


## Step 6: Show sweep table + winner


In [ ]:
from aggregate_sweep import write_sweep_table
payload = write_sweep_table()
print((paths.OUTPUT_ROOT_RESULTS / "sweep_comparison.md").read_text())
print("winning_config.json:")
print((paths.OUTPUT_ROOT_RESULTS / "winning_config.json").read_text())
